# Solution 2.5: Merging & Combining Datasets (Angola IEA and INE trade)

Three separate join problems, on purpose:

- **Part A** attaches province names to the survey with a lookup table, using a
  deliberately outdated lookup so the audit tools have something to find.
- **Part B** appends the Q3 2025 wave of the same survey to the Q4 wave, where
  the two raw files share only 20 of their 260 and 206 column names, and after
  a22 renamed the Q4 columns to English even that overlap drops to zero.
- **Part C** merges Angola's export and import tables to compute a trade balance.

Part C uses a different dataset. There is no sensible join between an individual
level labour survey and country level trade totals, and pretending otherwise
would teach a bad habit.

> **Pipeline:** run Exercise 2.4 first. Writes three files to `20_processed/`.

### Path Setup (run first)

In [1]:
import os

import numpy as np
import pandas as pd

DATA_PROC_DIR = '../../data/20_processed'
DATA_SURVEY_DIR = '../../data/0_raw/angola/employment_survey'
DATA_TRADE_DIR = '../../data/0_raw/angola/international_trade'

features_path = os.path.join(DATA_PROC_DIR, 'angola_iea_2025q4_features.csv')

STR_COLS = {
    'household_id': 'string', 'person_no': 'string',
    'cluster_id': 'string', 'province_code': 'string',
}
df = pd.read_csv(features_path, dtype=STR_COLS)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Survey:', df.shape)
df[['household_id', 'province_code', 'age', 'lf_status_strict']].head()

Survey: (53297, 39)


,household_id,province_code,age,lf_status_strict
0,10000003,25,55.00,Outside labour force
1,10000003,25,52.00,Outside labour force
2,10000003,25,19.00,Outside labour force
3,10000003,25,16.00,Unemployed
4,10000009,25,43.00,Employed


---

# Part A: attaching province names with a lookup

## Task 1: A lookup table, as published before 2024

Angola reorganised its provinces in 2024, going from 18 to 21. Icolo e Bengo,
Moxico Leste and Cuando are new. The lookup below is the **old** 18 province
list, which is exactly what you get if you copy a reference table from an older
publication.

In [2]:
province_lookup_old = pd.DataFrame({
    'province_code': ['10', '11', '12', '13', '14', '15', '16', '17', '18',
                      '19', '20', '21', '22', '23', '24', '25', '26', '27'],
    'province_name_ref': ['Cabinda', 'Zaire', 'Uíge', 'Bengo', 'Luanda',
                          'Cuanza-Norte', 'Cuanza-Sul', 'Malanje', 'Lunda-Norte',
                          'Lunda-Sul', 'Moxico', 'Bié', 'Huambo', 'Benguela',
                          'Namibe', 'Huila', 'Cunene', 'Cubango'],
})
print('Lookup rows:', len(province_lookup_old))
province_lookup_old.head()

Lookup rows: 18


,province_code,province_name_ref
0,10,Cabinda
1,11,Zaire
2,12,Uíge
3,13,Bengo
4,14,Luanda


## Task 2: Join key hygiene, before you merge

Most failed merges are a key that is text on one side and a number on the other,
or a stray space. Check dtypes, whitespace and missing keys on both sides first.

In [3]:
print('Survey key dtype:', df['province_code'].dtype)
print('Lookup key dtype:', province_lookup_old['province_code'].dtype)
print()
print('Missing keys, survey:', df['province_code'].isna().sum())
print('Missing keys, lookup:', province_lookup_old['province_code'].isna().sum())
print('Duplicate keys, lookup:', province_lookup_old['province_code'].duplicated().sum())

Survey key dtype: string
Lookup key dtype: object

Missing keys, survey: 0
Missing keys, lookup: 0
Duplicate keys, lookup: 0


**Answers:**

- Both keys are `string` because 2.2 converted the survey key deliberately. Had it
  stayed `float64`, every single row would fail to match a text lookup and the
  merge would return all `NaN` without any error at all.
- No missing keys on either side, and no duplicate keys in the lookup, so a left
  join should not change the row count.

## Task 3: Merge, then audit with `indicator`

A left join keeps every survey row. `indicator=True` adds a `_merge` column
labelling each row `both`, `left_only` or `right_only`, which is how you find out
what silently failed to match.

In [4]:
merged = pd.merge(df, province_lookup_old, on='province_code',
                  how='left', indicator=True)

print('Rows:', len(df), '->', len(merged))
print(merged['_merge'].value_counts())

Rows: 53297 -> 53297
_merge
both          46798
left_only      6499
right_only        0
Name: count, dtype: int64


In [5]:
unmatched = merged[merged['_merge'] == 'left_only']
print('Unmatched rows:', len(unmatched),
      f'({len(unmatched) / len(merged) * 100:.1f}%)')
print()
print(unmatched['province_code'].value_counts().sort_index())

Unmatched rows: 6499 (12.2%)

province_code
28    2883
29    2123
30    1493
Name: count, dtype: int64


In [6]:
inner = pd.merge(df, province_lookup_old, on='province_code', how='inner')
outer = pd.merge(df, province_lookup_old, on='province_code', how='outer')
print('inner:', len(inner), '| left:', len(merged), '| outer:', len(outer))

inner: 46798 | left: 53297 | outer: 53297


**Answers:**

- The row count is unchanged at 53,297, which is what a left join on a unique
  right key must do.
- **6,499 rows, 12.2%, are `left_only`**: province codes 28, 29 and 30. Those are
  the provinces created in 2024 and missing from the old lookup. Nothing raised an
  error; without `indicator=True` you would have shipped a table with 12% of the
  country silently unlabelled.
- The inner join drops those 6,499 rows entirely, which is worse: the data
  disappears rather than being visibly blank.
- The right response is to fix the lookup at source, not to drop the rows or
  invent names. In 2.4 the full 21 province map was already used successfully, so
  `province_name` is correct in this file and `province_name_ref` is the broken
  one. Comparing the two is the fastest way to prove a lookup is stale.

## Task 4: Cardinality, and making the assumption explicit

If the right hand key is not unique, every duplicate match multiplies rows.
`validate=` states your assumption and raises instead of silently inflating.

In [7]:
# A lookup that accidentally lists Cabinda twice
bad_lookup = pd.concat([province_lookup_old, province_lookup_old.head(1)],
                       ignore_index=True)

exploded = pd.merge(df, bad_lookup, on='province_code', how='left')
print('Rows before:', len(df), '-> after the bad merge:', len(exploded))
print('Extra rows:', len(exploded) - len(df))

Rows before: 53297 -> after the bad merge: 55602
Extra rows: 2305


In [8]:
try:
    pd.merge(df, bad_lookup, on='province_code', how='left', validate='many_to_one')
except Exception as error:
    print(type(error).__name__, '->', error)

MergeError -> Merge keys are not unique in right dataset; not a many-to-one merge


**Answers:**

- The row count grows from 53,297 to 55,602. Every person in Cabinda is duplicated
  because Cabinda appears twice on the right.
- `validate='many_to_one'` raises `MergeError`. Many survey rows to one lookup row
  is the correct description of a person to province join.
- `one_to_one` promises both sides are unique, `one_to_many` that the left is
  unique, `many_to_one` that the right is. Stating it converts a silent data
  corruption into an immediate, loud failure.

## Task 5: Post merge validation, then save

A merge is finished when you have confirmed the result, not when the code ran.

In [9]:
final = pd.merge(df, province_lookup_old, on='province_code', how='left')

print('Row count:', len(df), '->', len(final))
print('Duplicate person keys:',
      final.duplicated(subset=['household_id', 'person_no']).sum())
print('Unmatched province rate:', round(final['province_name_ref'].isna().mean(), 4))

Row count:

 53297 -> 53297


Duplicate person keys: 0
Unmatched province rate: 0.1219


In [10]:
final = final.reset_index(drop=True)
out_path = os.path.join(DATA_PROC_DIR, 'angola_iea_2025q4_analysis.csv')
final.to_csv(out_path, index=False)
print('Saved:', out_path, '|', final.shape)

Saved: ../../data/20_processed/angola_iea_2025q4_analysis.csv | (53297, 40)


**Answers:**

- 53,297 rows in and out, no duplicate person keys, and a 0.122 unmatched rate
  that we can explain exactly.
- An unmatched rate you can explain is fine to ship with a documented caveat. An
  unmatched rate you cannot explain is a stop sign.

---

# Part B: appending the Q3 and Q4 waves

## Task 6: Load the previous quarter

`IEA_III_TRIMESTRE_2025.sav` is the same survey, one quarter earlier. It uses the
questionnaire's own variable names rather than the ILO mnemonics of the Q4 file,
so almost nothing lines up.

In [11]:
q3_path = os.path.join(DATA_SURVEY_DIR, 'IEA_III_TRIMESTRE_2025.sav')
q3_full = pd.read_spss(q3_path, convert_categoricals=False)

print('Q3:', q3_full.shape)
print('Q4 features:', df.shape)
print('Column names in common:', len(set(q3_full.columns) & set(df.columns)))

Q3: (54963, 260)
Q4 features: (53297, 39)
Column names in common: 0


## Task 7: What a naive `concat` does

`pd.concat` aligns on column names and fills every gap with `NaN`, without a
single warning. Try it and measure the damage.

In [12]:
naive = pd.concat([q3_full, df], ignore_index=True)

print('Naive concat:', naive.shape)
mostly_empty = (naive.isna().mean() > 0.99).sum()
print(f'Columns more than 99% empty: {mostly_empty} of {naive.shape[1]}')
naive.iloc[:3, :6]

Naive concat: (108260, 299)


Columns more than 99% empty: 115 of 299


,NIDF,PROV,AREA_RESID,G_06_ID_IEA,G_06A_CS_SER_NUM,MUNIC
0,"3,370,096.00",14.00,1.00,337.00,"15,707.00","1,421.00"
1,"3,370,029.00",14.00,1.00,337.00,"15,787.00","1,421.00"
2,"3,370,069.00",14.00,1.00,337.00,"15,787.00","1,421.00"


**Answers:**

- The result is 108,260 rows by **299 columns**, of which 115 are more than 99%
  empty: Q3's 260 columns plus the Q4 feature table's 39, with **zero** names in
  common. Each wave contributed its own vocabulary and neither filled the other's.
- The two raw files do share 20 column names, but you are not stacking raw files
  here: a22 renamed the Q4 columns to English, so even that small overlap is gone.
  Renaming for readability quietly destroyed the only alignment that existed.
- Nothing warned. `concat` does exactly what it was asked; the mistake was in the
  asking.
- The row count is right and everything else is wrong, which is the dangerous
  kind of failure: it looks like it worked.

## Task 8: Do it properly, by harmonising first

Pick the variables that exist in both waves, rename the Q3 ones to the Q4 names,
confirm the two frames have identical columns, then stack them with a `wave`
column so no row loses its origin.

In [13]:
Q3_RENAME = {
    'NIDF': 'household_id', 'PROV': 'province_code', 'AREA_RESID': 'area_type',
    'S02_01': 'sex', 'S02_02': 'age', 'S4_01': 'worked_for_pay',
    'S4_02': 'worked_own_account', 'S4_03': 'worked_family_business',
    'S4_09': 'absent_from_job', 'S8_01': 'sought_work', 'S8_12': 'available_now',
    'POND_IEA_III_TRIM_2025_IND': 'weight_ind',
}

q3 = q3_full[list(Q3_RENAME)].rename(columns=Q3_RENAME)
q3['household_id'] = q3['household_id'].astype('int64').astype('string')
q3['province_code'] = q3['province_code'].astype('int64').astype('string').str.zfill(2)

print('Q3 harmonised:', q3.shape)
q3.head()

Q3 harmonised: (54963, 12)


,household_id,province_code,area_type,sex,age,worked_for_pay,worked_own_account,worked_family_business,absent_from_job,sought_work,available_now,weight_ind
0,3370096,14,1.00,2.00,88.00,2.00,2.00,2.00,2.00,2.00,2.00,"26,050.63"
1,3370029,14,1.00,1.00,15.00,2.00,2.00,2.00,2.00,2.00,2.00,"19,825.42"
2,3370069,14,1.00,1.00,18.00,2.00,2.00,2.00,2.00,2.00,1.00,"19,825.42"
3,3370002,14,1.00,2.00,12.00,NaN,NaN,NaN,NaN,NaN,NaN,"16,487.18"
4,3370002,14,1.00,2.00,7.00,NaN,NaN,NaN,NaN,NaN,NaN,"16,487.18"


In [14]:
SHARED = [
    'household_id', 'province_code', 'area_type', 'sex', 'age',
    'worked_for_pay', 'worked_own_account', 'worked_family_business',
    'absent_from_job', 'sought_work', 'available_now', 'weight_ind',
]

q3_slim = q3[SHARED].assign(wave='2025Q3')
q4_slim = df[SHARED].assign(wave='2025Q4')

print('Columns identical:', list(q3_slim.columns) == list(q4_slim.columns))

waves = pd.concat([q3_slim, q4_slim], ignore_index=True)
print('Stacked:', waves.shape)
print('Any column entirely empty:', waves.isna().all().any())
print(waves['wave'].value_counts())

Columns identical: True
Stacked: (108260, 13)
Any column entirely empty: False
wave
2025Q3    54963
2025Q4    53297
Name: count, dtype: int64


**Answers:**

- 108,260 rows by 13 columns instead of 299, and no column is empty.
- The `wave` column is added **before** stacking, so every row carries its origin.
  Without it the two quarters are indistinguishable and the append is
  irreversible.
- The cost is real: 12 usable variables, chosen by hand, out of 206 and 260.
  Note the three different counts in play: 20 raw names coincide, 0 survive the
  Q4 rename, and 12 are recoverable once you map them deliberately. Harmonising across
  waves means analysing the intersection, and the intersection is small.

## Task 9: Cross wave sanity checks

Two independent samples of the same population should agree on the things that do
not change quickly. If they do not, the append is wrong.

In [15]:
population = waves.groupby('wave')['weight_ind'].sum()
print('Weighted population by wave:')
print(population.round(0))
difference = abs(population.iloc[0] - population.iloc[1]) / population.iloc[1] * 100
print(f'Relative difference: {difference:.2f}%')

Weighted population by wave:
wave
2025Q3   37,337,629.00
2025Q4   37,604,687.00
Name: weight_ind, dtype: float64
Relative difference: 0.71%


In [16]:
# The harmonised definition can only use the variables both waves carry
for wave in ['2025Q3', '2025Q4']:
    sample = waves[waves['wave'] == wave]
    working_age = sample['age'] >= 15
    employed = working_age & (
        (sample['worked_for_pay'] == 1)
        | (sample['worked_own_account'] == 1)
        | (sample['absent_from_job'] == 1)
    )
    unemployed = (working_age & ~employed
                  & (sample['sought_work'] == 1)
                  & (sample['available_now'] == 1))
    weights = sample['weight_ind']
    rate = weights[unemployed].sum() / weights[employed | unemployed].sum() * 100
    print(f'{wave} harmonised strict unemployment: {rate:.1f}%')

2025Q3 harmonised strict unemployment: 11.5%
2025Q4 harmonised strict unemployment: 11.9%


In [17]:
out_path = os.path.join(DATA_PROC_DIR, 'angola_iea_waves_q3_q4.csv')
waves.to_csv(out_path, index=False)
print('Saved:', out_path, '|', waves.shape)

Saved: ../../data/20_processed/angola_iea_waves_q3_q4.csv | (108260, 13)


**Answers:**

- The weighted populations are 37,337,629 and 37,604,687, **0.71% apart**. Two
  independent samples agreeing that closely on the size of Angola is strong
  evidence that both the weights and the append are sound.
- Harmonised strict unemployment is 11.5% in Q3 and 11.9% in Q4: a small, credible
  quarter on quarter move.
- Note that Q4's harmonised 11.9% differs from the 14.5% computed in 2.4. Nothing
  is broken. The harmonised version can only use `sought_work` and
  `available_now`, because Q3 has no equivalent of `sought_business` or
  `available_2wk`. Comparability across waves costs precision within a wave, and
  that trade is the whole difficulty of producing a time series.
- Had the populations differed by 30%, the likely cause would be a weight column
  from the wrong wave, or a wave stacked twice.

---

# Part C: Angola's trade balance

Different dataset, different join. `Comercio Externo de Bens por Países
Parceiros.xlsx` is published by INE with four sheets: exports and imports, each
in kwanzas and in US dollars.

The sheets are formatted for human readers, so loading them takes work: two title
rows above the header, a blank row, a `Total Geral` row, and a source footer at
the bottom.

In [18]:
trade_path = os.path.join(DATA_TRADE_DIR,
                          'Comercio Externo de Bens por Países Parceiros.xlsx')

print(pd.ExcelFile(trade_path).sheet_names)

['Exportação por Países (Kz)', 'Exportação por Países (USD)', 'Importação por Países (Kz)', 'Importação por Países (USD)']


In [19]:
# What the raw sheet looks like before any cleaning
pd.read_excel(trade_path, sheet_name='Exportação por Países (USD)',
              header=None, nrows=6).iloc[:, :5]

,0,1,2,3,4
0,Quadro n.º 1 - Valores de exportação de bens d...,NaN,NaN,NaN,NaN
1,U.M.: Em milhares de USD,NaN,NaN,NaN,NaN
2,Código,País,Ano\n2004,Ano\n2005,Ano\n2006
3,NaN,NaN,NaN,NaN,NaN
4,Total Geral,NaN,"13,353,207.48","23,633,561.86","31,755,033.64"
5,AF,Afeganistão,15.00,0,0


## Task 10: Load a sheet properly

`skiprows=2` puts the real header row in place. The country code must be read as
text, the header names carry an embedded newline, and the total and footer rows
both lack a country name, which makes them easy to remove together.

In [20]:
def load_trade_sheet(path, sheet):
    """Load one INE trade sheet and strip its title, total and footer rows."""
    frame = pd.read_excel(path, sheet_name=sheet, skiprows=2, dtype={'Código': str})
    frame.columns = frame.columns.str.replace('\n', ' ', regex=False).str.strip()
    frame = frame[frame['País'].notna()].copy()
    return frame.rename(columns={'Código': 'country_code', 'País': 'country_name'})


exports = load_trade_sheet(trade_path, 'Exportação por Países (USD)')
imports = load_trade_sheet(trade_path, 'Importação por Países (USD)')

print('Exports:', exports.shape, '| Imports:', imports.shape)
print('Columns:', list(exports.columns)[:4], '...', list(exports.columns)[-2:])
exports.head()

Exports: (249, 24) | Imports: (249, 24)
Columns: ['country_code', 'country_name', 'Ano 2004', 'Ano 2005'] ... ['Ano 2024', 'Ano 2025']


,country_code,country_name,Ano 2004,Ano 2005,Ano 2006,Ano 2007,Ano 2008,Ano 2009,Ano 2010,Ano 2011,...,Ano 2016,Ano 2017,Ano 2018,Ano 2019,Ano 2020,Ano 2021,Ano 2022,Ano 2023,Ano 2024,Ano 2025
2,AF,Afeganistão,15.00,0.00,0.00,0.00,0.04,0.36,14.35,11.53,...,3.06,2.25,20.89,0.00,0.00,0.00,0.00,0.09,0.00,0.41
3,ZA,África do Sul,"169,993.75","348,507.33","549,722.61","1,808,430.04","2,514,406.28","1,439,776.01","1,734,082.44","1,707,942.48",...,"1,309,198.41","1,342,919.68","1,130,056.26","365,004.93","180,143.03","512,185.51","294,538.75","435,221.88","822,693.60","593,519.94"
4,AL,Albânia,3.71,0.00,1.99,1.60,0.00,41.61,0.00,0.00,...,0.00,39.72,2.71,48.57,0.00,0.00,65.67,0.00,0.00,2.52
5,DE,Alemanha,"1,569.58","1,635.21","2,732.33","2,543.48","127,182.28","70,654.96","6,937.47","125,221.39",...,"16,126.84","8,025.81","2,693.01","4,446.17","4,152.40","4,202.87","8,368.71","297,893.69","50,975.25","55,864.20"
6,AD,Andorra,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,5.98,0.50,0.05,206.63,154.58,0.00,7.06,0.92,3.90


**Answers:**

- Each sheet gives 249 rows: 248 countries plus `ZZ`, Desconhecido, meaning the
  partner was not recorded.
- The single filter `frame['País'].notna()` removes the blank row, the
  `Total Geral` row and the `Fonte: INE` footer in one step, because none of them
  has a country name. Dropping rows by position would break the moment INE adds a
  line.
- `dtype={'Código': str}` keeps codes as text. It matters for the same reason it
  mattered for province codes: they are labels, not quantities.
- The newline inside `Ano\n2004` is invisible when printed but would make every
  later column reference fail.

## Task 11: Merge exports against imports

Both tables have one row per country, so this is a one to one merge. An outer
join keeps partners that appear on only one side.

In [21]:
YEAR = 'Ano 2025'

trade = pd.merge(
    exports[['country_code', 'country_name', YEAR]].rename(columns={YEAR: 'exports_usd'}),
    imports[['country_code', YEAR]].rename(columns={YEAR: 'imports_usd'}),
    on='country_code',
    how='outer',
    indicator=True,
    validate='one_to_one',
)
print('Merged:', trade.shape)
print(trade['_merge'].value_counts())

Merged: (249, 5)
_merge
both          249
left_only       0
right_only      0
Name: count, dtype: int64


In [22]:
trade['balance_usd'] = trade['exports_usd'].fillna(0) - trade['imports_usd'].fillna(0)

print('Largest surpluses:')
print(trade.nlargest(5, 'balance_usd')[['country_name', 'exports_usd',
                                        'imports_usd', 'balance_usd']].to_string(index=False))
print()
print('Largest deficits:')
print(trade.nsmallest(5, 'balance_usd')[['country_name', 'exports_usd',
                                         'imports_usd', 'balance_usd']].to_string(index=False))

Largest surpluses:
          country_name   exports_usd  imports_usd   balance_usd
                 China 14,450,839.72 3,420,626.27 11,030,213.45
                 Índia  3,639,895.11 1,043,808.31  2,596,086.79
             Indonésia  1,967,048.85   146,711.05  1,820,337.80
               Espanha  1,721,616.77   284,839.90  1,436,776.87
Emirados Árabes Unidos  1,722,331.20   683,052.59  1,039,278.61

Largest deficits:
             country_name  exports_usd  imports_usd   balance_usd
                 Portugal   169,020.79 1,642,901.16 -1,473,880.37
              Reino Unido   172,270.94 1,002,682.20   -830,411.26
            Coreia do Sul        70.40   805,860.19   -805,789.79
                Argentina       188.24   569,191.09   -569,002.85
Estados Unidos da América   393,855.41   925,961.70   -532,106.29


In [23]:
print(trade[trade['country_code'] == 'ZZ'][
    ['country_code', 'country_name', 'exports_usd', 'imports_usd', 'balance_usd']])

    country_code  country_name  exports_usd  imports_usd  balance_usd
247           ZZ  Desconhecido   229,897.39         1.25   229,896.14


**Answers:**

- All 249 countries are `both`, and `validate='one_to_one'` passed, so each
  partner appears exactly once on each side. Confirming a clean merge is a
  result, not a wasted check.
- China is by far the largest surplus partner and Portugal the largest deficit,
  which matches Angola's oil export and consumer goods import profile.
- `ZZ`, Desconhecido, carries 229,897 thousand USD of exports with an unrecorded
  partner. It is not an error to delete: it is a real, quantified gap in the trade
  statistics, and it belongs in a footnote. Silently dropping it would make the
  export total wrong.
- `fillna(0)` before subtracting is a decision, not a formality. It treats "no
  trade recorded" as zero trade, which is reasonable here and would not be if the
  gap meant "not yet reported".

## Task 12: Save the trade table

In [24]:
trade = trade.drop(columns='_merge').reset_index(drop=True)
out_path = os.path.join(DATA_PROC_DIR, 'angola_trade_partners.csv')
trade.to_csv(out_path, index=False)
print('Saved:', out_path, '|', trade.shape)

Saved: ../../data/20_processed/angola_trade_partners.csv | (249, 5)


**Answers:**

- Three files written by this notebook: the province joined survey, the two wave
  stack, and the trade balance.
- Merging combines columns and needs a key. Appending combines rows and needs a
  shared schema. Part A and Part C did the first, Part B did the second, and the
  hard part in every case was the checking rather than the call itself.